# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [1]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Eden-Sibhat/2026-HYU-AUE8088-PA2

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 103 (delta 21), reused 12 (delta 9), pack-reused 70 (from 2)
Receiving objects: 100% (103/103), 202.58 MiB | 18.41 MiB/s, done.
Resolving deltas: 100% (42/42), done.
Updating files: 100% (35/35), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.resnet import resnet18

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "top1k-uncertainty"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5", STRATEGY_NAME]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: edensibhat (edensibhat-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=9658227a-10cf-4ff9-bcee-0ee4cd4936b6
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 196MB/s]


압축 해제 중...
완료 → ../data/set_a


In [6]:
# 1단계 — best base 모델로 Set B 의 모든 이미지를 score.
model = resnet18().to(device)

ckpt = torch.load(
    "checkpoints/level3_best.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(ckpt["state_dict"])
model.eval()

print("Loaded Level 3 checkpoint")
print("Best Avg-Macro-F1:", ckpt.get("best_avg_mf1", "not saved"))

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성 계산: 1 - max-softmax 를 3개 head 평균
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)

print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")

Loaded Level 3 checkpoint
Best Avg-Macro-F1: 0.6530525039583961
unc shape: (15000,), mean=0.296


In [8]:
# 2단계 —
import json
import numpy as np
from collections import Counter

K = 1000

# --------------------------------------------------
# 1. Set B 정답 라벨 가져오기
# --------------------------------------------------
labels_b = {a: [] for a in ATTRIBUTES}

for s in set_b.samples:
    labels_b["weather"].append(int(s.weather))
    labels_b["scene"].append(int(s.scene))
    labels_b["timeofday"].append(int(s.timeofday))

for a in ATTRIBUTES:
    labels_b[a] = np.array(labels_b[a])


# --------------------------------------------------
# 2. Hard example score
#    모델 예측이 틀린 task가 많을수록 높은 점수
# --------------------------------------------------
hard_score = np.zeros(len(set_b))

for a in ATTRIBUTES:
    hard_score += (preds_b[a] != labels_b[a]).astype(float)

hard_score = hard_score / len(ATTRIBUTES)


# --------------------------------------------------
# 3. Rare class score
#    Set A train에서 적게 나온 class일수록 높은 점수
# --------------------------------------------------
set_a_train = BDDAttrDataset("../data/set_a", split="train", transform=eval_transform())

class_counts = {a: Counter() for a in ATTRIBUTES}

for s in set_a_train.samples:
    class_counts["weather"][int(s.weather)] += 1
    class_counts["scene"][int(s.scene)] += 1
    class_counts["timeofday"][int(s.timeofday)] += 1

rare_score = np.zeros(len(set_b))

for i in range(len(set_b)):
    score = 0.0

    for a in ATTRIBUTES:
        cls = labels_b[a][i]
        count = class_counts[a][cls]

        # count가 작을수록 rare score가 커짐
        score += 1.0 / np.sqrt(count + 1)

    rare_score[i] = score / len(ATTRIBUTES)


# --------------------------------------------------
# 4. Normalize scores to 0~1
# --------------------------------------------------
def normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

uncertainty_norm = normalize(uncertainty)
hard_score_norm = normalize(hard_score)
rare_score_norm = normalize(rare_score)


# --------------------------------------------------
# 5. Final curation score
# --------------------------------------------------
final_score = (
    0.4 * uncertainty_norm +
    0.3 * hard_score_norm +
    0.3 * rare_score_norm
)

order = np.argsort(-final_score)[:K]


# --------------------------------------------------
# 6. Save selected 1000 images
# --------------------------------------------------
picks = []

for i in order:
    s = set_b.samples[i]

    reason_list = []

    if uncertainty_norm[i] > 0.7:
        reason_list.append("high uncertainty")

    if hard_score_norm[i] > 0.5:
        reason_list.append("hard example")

    if rare_score_norm[i] > 0.7:
        reason_list.append("rare class")

    if len(reason_list) == 0:
        reason_list.append("high combined score")

    picks.append({
        "image_id": s.image_id,
        "weather": int(s.weather),
        "scene": int(s.scene),
        "timeofday": int(s.timeofday),
        "score": float(final_score[i]),
        "uncertainty_score": float(uncertainty_norm[i]),
        "hard_score": float(hard_score_norm[i]),
        "rare_score": float(rare_score_norm[i]),
        "reason": ", ".join(reason_list),
    })


with open("level5_picks.json", "w") as f:
    json.dump({
        "strategy": (
            "Hybrid active data curation using uncertainty, hard-example mining, "
            "and rare-class balancing. Uncertainty selects images where the model has low confidence, "
            "hard-example mining selects images predicted incorrectly by the Level 3 model, "
            "and rare-class balancing gives priority to underrepresented classes in Set A. "
            "The final 1000 images are selected using a weighted score: "
            "0.4 uncertainty + 0.3 hard-example + 0.3 rare-class."
        ),
        "num_picks": len(picks),
        "weights": {
            "uncertainty": 0.4,
            "hard_example": 0.3,
            "rare_class": 0.3,
        },
        "picks": picks,
    }, f, indent=2)

print(f"saved {len(picks)} picks to level5_picks.json")
print("First 5 picks:")
for p in picks[:5]:
    print(p)

saved 1000 picks to level5_picks.json
First 5 picks:
{'image_id': '27b3b7a0-7586c958', 'weather': 3, 'scene': 2, 'timeofday': 2, 'score': 0.9935752153396606, 'uncertainty_score': 0.9839382171630859, 'hard_score': 1.0, 'rare_score': 0.9999997019767761, 'reason': 'high uncertainty, hard example, rare class'}
{'image_id': '2ad035f4-ce3be4f4', 'weather': 3, 'scene': 2, 'timeofday': 2, 'score': 0.9164026975631714, 'uncertainty_score': 0.7910069823265076, 'hard_score': 1.0, 'rare_score': 0.9999997019767761, 'reason': 'high uncertainty, hard example, rare class'}
{'image_id': '5a30a603-211bdb88', 'weather': 3, 'scene': 0, 'timeofday': 2, 'score': 0.8922443985939026, 'uncertainty_score': 0.8991841673851013, 'hard_score': 1.0, 'rare_score': 0.7752355933189392, 'reason': 'high uncertainty, hard example, rare class'}
{'image_id': '4b692d6a-e4458dfd', 'weather': 3, 'scene': 2, 'timeofday': 2, 'score': 0.8752270936965942, 'uncertainty_score': 0.6880677342414856, 'hard_score': 1.0, 'rare_score': 0.9

In [10]:
# 3단계 — Set A + 본인이 고른 picks 로 재학습. 학습 메트릭은 wandb 로 자동 로깅.

import os
import torch
from collections import Counter

STRATEGY_NAME = "hybrid_uncertainty_hard_rare"

os.makedirs("checkpoints", exist_ok=True)

extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]

train_aug = BDDAttrDataset(
    "../data/set_a",
    "train",
    transform=train_transform(),
    extra_picks=extra,
)

val_ds = BDDAttrDataset(
    "../data/set_a",
    "val",
    transform=eval_transform(),
)

g = torch.Generator()
g.manual_seed(SEED)

loader_tr = DataLoader(
    train_aug,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

loader_val = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

set_seed(SEED, deterministic=True)

model2 = resnet18().to(device)

optim = torch.optim.AdamW(
    model2.parameters(),
    lr=3e-4,
    weight_decay=5e-4,
)

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim,
    T_max=25,
)

losses = {
    a: nn.CrossEntropyLoss()
    for a in ATTRIBUTES
}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "backbone": "resnet18",
        "strategy": STRATEGY_NAME,
        "num_picks": len(picks),
        "epochs": 25,
        "batch": 64,
        "lr": 3e-4,
        "seed": SEED,
    },
    tags=WANDB_TAGS + ["level5", STRATEGY_NAME],
)

trainer = MultiTaskTrainer(
    model2,
    optim,
    sched,
    losses,
    device,
    TrainConfig(epochs=25),
    logger=logger,
)

history = trainer.fit(loader_tr, loader_val)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/25] train_loss=2.4474  val_avg_MF1=0.4490  per={'weather': 0.21273670139518638, 'scene': 0.3300764865531705, 'timeofday': 0.8041704523186004}


[epoch 02/25] train_loss=2.2405  val_avg_MF1=0.4730  per={'weather': 0.2672079446902654, 'scene': 0.38181890177613625, 'timeofday': 0.7699470690850001}


[epoch 03/25] train_loss=2.1755  val_avg_MF1=0.4777  per={'weather': 0.25574470856897635, 'scene': 0.37819343932227856, 'timeofday': 0.7991247926574293}


[epoch 04/25] train_loss=2.1210  val_avg_MF1=0.5505  per={'weather': 0.4345444251426714, 'scene': 0.4301389037163565, 'timeofday': 0.786949715650457}


...

[epoch 21/25] train_loss=1.4673  val_avg_MF1=0.6481  per={'weather': 0.5204110355626941, 'scene': 0.5904501710953324, 'timeofday': 0.8334366476470461}


[epoch 22/25] train_loss=1.4260  val_avg_MF1=0.6270  per={'weather': 0.5343188974486893, 'scene': 0.5692878430206262, 'timeofday': 0.7773022306999416}


[epoch 23/25] train_loss=1.4182  val_avg_MF1=0.6670  per={'weather': 0.5401992363765109, 'scene': 0.6404529675485889, 'timeofday': 0.8203431171751641}


[epoch 24/25] train_loss=1.3908  val_avg_MF1=0.6547  per={'weather': 0.5386494356980468, 'scene': 0.6180982440499728, 'timeofday': 0.8072495867032821}


[epoch 25/25] train_loss=1.3827  val_avg_MF1=0.6623  per={'weather': 0.5481269075539649, 'scene': 0.6375551087065066, 'timeofday': 0.801114891308686}


In [11]:
# 학습 종료 후 — 속성별 confusion matrix 와 picks 분포를 wandb 에 업로드

val_metrics = trainer.evaluate(loader_val)

print("Level 5 final validation metrics:")
for k, v in val_metrics.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

# 본인이 고른 1,000장의 분포
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    rows = [
        [CLASS_NAMES[a][k], cnt.get(k, 0)]
        for k in range(NUM_CLASSES[a])
    ]

    logger.log_table(
        f"picks/distribution_{a}",
        ["class", "count"],
        rows,
    )

torch.save(
    {
        "state_dict": model2.state_dict(),
        "history": history,
        "val_metrics": val_metrics,
        "strategy": STRATEGY_NAME,
        "num_picks": len(picks),
    },
    "checkpoints/level5_final.pth",
)

print("Saved Level 5 final model to checkpoints/level5_final.pth")

logger.finish()

Level 5 final validation metrics:
avg_macro_f1: 0.6623
per_macro_f1: {'weather': 0.5481269075539649, 'scene': 0.6375551087065066, 'timeofday': 0.801114891308686}
preds: {'weather': array([0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 5, 0, 0, 0, 0, 0,
       0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3,
       0, 0, 0, 

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁
val/avg_macro_f1,▁▂▂▄▂▅▅▅▄▅▄▅▅▅▆▇█▆█▆▇▇███
val/mf1_scene,▁▂▂▃▃▄▄▂▄▄▃▃▄▅▅▇█▅▇▅▇▆█▇█
val/mf1_timeofday,▄▂▄▃▁▅▄█▅▇▆▆▅▄▆▆▄▆▅▄▆▂▅▄▄
val/mf1_weather,▁▂▂▆▃▅▆▇▄▅▅▅▅▅▇▆█▆█▇▇████
epoch,25
lr,0
train/loss,1.38267
val/avg_macro_f1,0.66227


In [13]:
# 4단계 — Kaggle 제출용 CSV 생성.
model2.eval()
test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.


In [14]:
!ls -lh ../submission/

total 24K
-rw-r--r-- 1 root root 22K Jun 22 08:59 level5_submission.csv


In [15]:
from google.colab import files
files.download("../submission/level5_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.